# 03 -- Strategy/regime analysis

Paired scripts: `analysis/regime_validation.py` (a Python port of `MarketRegimeEngine.mqh`'s
(TASK-016) classification FORMULA -- see that module's own docstring for why this does
**NOT** close TASK-016's deferred item: only 7 of 9 states, no gating/hysteresis, no
real-evidence confusion matrix; full completion is tracked separately as
`TASK-031_REGIME_VALIDATION_COMPLETION.md`) and `analysis/performance_breakdown.py` (the
real, paired, tested strategy+regime performance-grouping pipeline -- **fixed, 2026-07-22
Codex review finding:** this notebook previously computed a regime-only breakdown INLINE,
with no strategy/setup dimension at all, calling it "strategy-performance analysis" despite
never grouping by strategy -- a violation of the reproducibility contract's paired-pipeline
rule. `performance_breakdown.py` now does the real grouping; this notebook only builds the
fixture and calls it).

Demonstrates ALL SEVEN directly-computed regime states against hand-constructed synthetic
fixtures (the same ones hand-verified in `tests/test_regime_validation.py`), then joins a
synthetic per-trade strategy+regime label onto synthetic trade outcomes via
`performance_breakdown.compute_breakdown` to show performance BY STRATEGY AND REGIME.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.performance_breakdown import compute_breakdown
from analysis.regime_validation import Regime, build_confusion_matrix, classify

TREND_CLOSES = [104.0, 103.0, 102.0, 101.0, 100.0]
CHOPPY_CLOSES = [100.0, 101.0, 100.0, 101.0, 100.0]
COMMON = dict(
    efficiency_window=4,
    trend_threshold=0.6,
    expansion_threshold=0.75,
    compression_threshold=0.25,
    min_efficiency=0.3,
    trend_slope_atr_divisor=0.5,
)

In [ ]:
# All 7 directly-computed regime states (the same fixtures hand-verified
# in tests/test_regime_validation.py, reproduced here in full -- not a
# subset).
results = {
    "TRENDING_UP": classify(
        TREND_CLOSES,
        [0.0, 2.0, 2.0],
        1.0,
        ema_now=105.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=1.0,
        direction_agree=True,
    ),
    "TRENDING_DOWN": classify(
        TREND_CLOSES,
        [0.0, 2.0, 2.0],
        1.0,
        ema_now=95.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=1.0,
        direction_agree=True,
    ),
    "VOLATILITY_EXPANSION_UP": classify(
        TREND_CLOSES,
        [0.0, 1.0, 1.0],
        2.0,
        ema_now=105.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=1.0,
        direction_agree=True,
    ),
    "VOLATILITY_EXPANSION_DOWN": classify(
        TREND_CLOSES,
        [0.0, 1.0, 1.0],
        2.0,
        ema_now=95.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=1.0,
        direction_agree=True,
    ),
    "TRANSITION_OR_UNCERTAIN": classify(
        TREND_CLOSES,
        [0.0, 1.0, 1.0],
        2.0,
        ema_now=105.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=0.0,
        direction_agree=False,
    ),
    "COMPRESSION": classify(
        TREND_CLOSES,
        [0.0, 5.0, 5.0],
        1.0,
        ema_now=100.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=0.0,
        direction_agree=False,
    ),
    "RANGING (efficiency failure)": classify(
        CHOPPY_CLOSES,
        [0.0, 0.5, 0.5],
        1.0,
        ema_now=100.0,
        ema_prior=100.0,
        adx_now=50.0,
        **COMMON,
        swing_agreement=0.0,
        direction_agree=False,
    ),
}

for label, r in results.items():
    print(
        f"{label:32s} -> regime={r.regime.value:28s} confidence={r.confidence:.4f} (valid={r.valid})"
    )

assert results["TRENDING_UP"].regime == Regime.TRENDING_UP
assert results["TRENDING_DOWN"].regime == Regime.TRENDING_DOWN
assert results["VOLATILITY_EXPANSION_UP"].regime == Regime.VOLATILITY_EXPANSION_UP
assert results["VOLATILITY_EXPANSION_DOWN"].regime == Regime.VOLATILITY_EXPANSION_DOWN
assert results["TRANSITION_OR_UNCERTAIN"].regime == Regime.TRANSITION_OR_UNCERTAIN
assert results["COMPRESSION"].regime == Regime.COMPRESSION
assert results["RANGING (efficiency failure)"].regime == Regime.RANGING

## Performance BY strategy AND regime (synthetic)

The master-prompt's own requirement is performance broken down by strategy/setup/regime/
session/etc. This cell demonstrates that shape of analysis using synthetic per-trade
strategy+regime labels via the real, paired `performance_breakdown.compute_breakdown`
pipeline (a real version would come from joining `regime_validation.classify` output and
the EA's own `strategy` journal field onto each trade's own decision timestamp, once real
journal + price data and the durable signal/order/deal identity join exist -- see
`TASK-036_JOURNAL_PRODUCER_COMPLETION.md`/`TASK-037_MT5_EXPORT_BRIDGE.md`).

In [ ]:
synthetic_trades = pd.DataFrame(
    {
        "trade_id": [f"t{i}" for i in range(10)],
        "strategy": ["SR_BOUNCE"] * 4 + ["SMC"] * 3 + ["TREND_FOLLOWING"] * 3,
        "regime": ["TRENDING_UP"] * 4 + ["RANGING"] * 3 + ["COMPRESSION"] * 3,
        "profit": [10.0, 10.0, -5.0, 10.0, -5.0, 10.0, -5.0, -5.0, -5.0, -5.0],
    }
)

# Real, paired pipeline call (analysis.performance_breakdown.compute_breakdown) --
# not an inline groupby -- grouping by BOTH strategy and regime together.
performance_by_strategy_regime = compute_breakdown(synthetic_trades, ["strategy", "regime"])
print(
    performance_by_strategy_regime[
        ["strategy", "regime", "n_trades", "win_rate", "expectancy_dollars"]
    ]
)

assert len(performance_by_strategy_regime) == 3
sr_bounce_row = performance_by_strategy_regime[
    performance_by_strategy_regime["strategy"] == "SR_BOUNCE"
].iloc[0]
assert sr_bounce_row["regime"] == "TRENDING_UP"
assert abs(sr_bounce_row["win_rate"] - 0.75) < 1e-9  # 3 wins of 4
trend_following_row = performance_by_strategy_regime[
    performance_by_strategy_regime["strategy"] == "TREND_FOLLOWING"
].iloc[0]
assert abs(trend_following_row["win_rate"] - 0.0) < 1e-9  # 0 wins of 3

In [ ]:
# Small illustrative confusion matrix (synthetic labels, not real evidence).
predicted = ["TRENDING_UP", "TRENDING_UP", "RANGING", "RANGING", "TRENDING_UP"]
actual = ["TRENDING_UP", "RANGING", "RANGING", "RANGING", "TRENDING_UP"]
matrix = build_confusion_matrix(predicted, actual)
print(matrix)

## Confusion matrix against real, independently-labelled evidence: PENDING

No such dataset exists yet in this project. `build_confusion_matrix` above is ready to use
once one does. Also note (see `regime_validation.py`'s own docstring):
`swing_agreement`/`direction_agree` were supplied directly here, not computed from real
`MarketStructure.mqh`-equivalent logic -- that port is separate, not-yet-attempted work.
Similarly, the performance-by-regime join above uses SYNTHETIC per-trade regime labels, not
a real join against journal/price data (no real journal exists yet either).